# Inflation Forecasting, data structure and model evaluation

# Analys 1: 12m rullande KPIF inflationsdata

# Ladda paket och data

In [ ]:
# Beräkningar
import pandas as pd
import numpy as np
import scipy.stats as stats

# Stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import (
    adfuller,
    acf,
    pacf
)

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import zivot_andrews
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import ttest_1samp

from pmdarima import auto_arima

# Plots
import plotly.express as px
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ARCH-paket
from arch.unitroot import ADF
from arch.unitroot import KPSS
from arch.unitroot import ZivotAndrews

In [ ]:
inf_data = pd.read_csv('../data/KPIF&OTHER.csv')

In [ ]:
inf_data = inf_data.rename(columns={
    'KPI med fast bostadsränta (KPIF)':'KPIF',
    'Harmoniserat KPI (HIKP)':'HIKP', 
    'KPI, fastst�llda tal (1980=100)':'KPI, fast tal (1980:100)'})
inf_data = inf_data.drop(columns={
    'enhet'
})

# Inspektera och justera datatyper

In [ ]:
print(inf_data.info())
print(inf_data.head())

In [ ]:
inf_data['period'] = pd.to_datetime(inf_data['period'], format='%YM%m')

In [ ]:
in_fig = px.line(inf_data,
                 x='period',
                 y='KPIF')
in_fig.show()

# Diffa -> inspektera -> skapa en trimmad serie

In [ ]:
inf_data['KPIF-12m-Change'] = inf_data['KPIF'].pct_change(periods=12)

In [ ]:
KPIF_12m_Ch = inf_data[['period','KPIF-12m-Change']].copy().dropna().reset_index(drop=True)

In [ ]:
# Sätter datum till index (anropas "KPIF_12m_ch.index")
KPIF_12m_Ch = KPIF_12m_Ch.set_index('period')

In [ ]:
KPIF_12m_Ch.head()

In [ ]:
# Otrimmad - från start till maj 2026
KPIF_full = KPIF_12m_Ch[['KPIF-12m-Change']].loc[:'2026-05-01'].dropna()

# Trimmad - från 1994 till maj 2026  
KPIF_trimmed = KPIF_12m_Ch[['KPIF-12m-Change']].loc['1994-01-01':'2026-05-01'].dropna()

In [ ]:
# Anger man inget x så används index som default.
# Plottar otrimmad serie
in_ch_fig = px.line(KPIF_full,
                 y='KPIF-12m-Change')
in_ch_fig.show()

Plottar 12mån rullande förändrng i KPIF.
Det är möjligt att jag hade kunnat ana att grafen skulle se ut såhär givet extremt få perioder med deflation.
Grafen ger en indikation om att vi behöver ta hänsyn till flera aspekter.
    - Trend
    - Trendbrott
    - Chocker

Så vi har två val. Använda oss av en modell som kan hantera de strukturella brotten, eller kapa serien. 
Från 1994 och framåt så har inflationen rört sig i en ganska tight range (bortstet från covid-chocken) och skulle därmed vara en tidsperiod som är lättare att modella.
    - Skulle kunna testa båda och sedan jämföra utfall

Jag kopierar serien KPIF_12m_Change och skapar en ny som har start 1994. Därefter så kör vi ett vanligt ADF-test på respektive serie.
    - De mer avancerade ADF-testerna används för att undvika typ 2 fel, att exempelvis en stationär serie med trendbrott inte är stationär trots att den är det i de två perioderna separerat av trendbrottet. 
        - Men i mitt fall så ser inte den första perioden ut att vara staionär alls.

In [ ]:
# Plottar trimmad serie
in_cht_fig = px.line(KPIF_trimmed,
                 y='KPIF-12m-Change')
in_cht_fig.show()

# Testa för enhetsrötter och stationäritet - ADF, Zivot-Andrews, KPSS

In [ ]:
# Väljer max-lags med formel kmax​=⌊12(T/100)1/4⌋ = 18,5 för T=564 och 17 för T=408
KPIF_adf = ADF(KPIF_full['KPIF-12m-Change'],
               trend='ct',
               method='bic',
               max_lags=18)
print(KPIF_adf.summary())

In [ ]:
KPIFt_adf = ADF(KPIF_trimmed['KPIF-12m-Change'],
               trend='ct',
               method='bic',
               max_lags=17)
print(KPIFt_adf.summary())

Varken hela eller trimmade serien ger ett ADF som förkastar H0. Detta är anmärkningsvärt då serier som inte är stationära har explosiv varians.
    - Detta betyder att variansen växer med horisonten och prognosintervallen exploderar och blir oprecisa.

Den trimmade serien var däremot inte långt ifrån att förkasta H0, och det skulle kunna ha varit en fråga om frihetsgrader och reducerad teststyrka (modellen var överspecificerad). Så vi kör om testet på den trimmade serien med enbart 'c' = konstant.

In [ ]:
KPIFt2_adf = ADF(KPIF_trimmed['KPIF-12m-Change'].dropna(),
               trend='c',
               method='bic',
               max_lags=17)
print(KPIFt2_adf.summary())

- Varför 12 lags? Och vad betyder dessa 12 lags?

Valet av 12 lags för testet görs på grund av att det tar 12 månader för autokorrelationen att dö ut i respektive datapunkt. Efter 12 lags har observationerna inga gemensama termer ut, vilket ät logiskt med tanke på att varje datapunkt representerar ett års inflationsförändring.

- Testet ovan på den trimmade serien förkastar h0

Det negativa test statsitics på -2,979<-2.87 är signifikant nog att förkasta på 5% nivå. Vilket är en bra indikation på att serien inte har en enhetsrot. H0 förkastas alltså till fördel för H1 och indikerar att serien är stationär.

H0: Serien har en enhetsrod. y = 0 där y = (p-1)
H1: Serien är stationär. Ingen enhetsrot. y > 0.

In [ ]:
# Vi kan köra ett KPSS test för stationäritet av robusthetsskäl. Inte nödvändigt, men kan användas som en "extra lens"
# när ADF-testet ger ett "osäkert" svar. exempelvis när man precis missar en signifikansnivå eller när man misstänker
# låg teststyrka.
# Väljer 12 lags här då ADF-testet indikerade att 12 var "bäst" pga autokorrelation dör ut.

KPIFt_KPSS = KPSS(KPIF_trimmed['KPIF-12m-Change'],
                  trend='c',
                  lags=12)
print(KPIFt_KPSS.summary())

KPSS-testet på den trimmade serien skapar lite bekymmer för oss. Med p-värde på 0,078 så förkastas H0 vid en signifikant nivå som 10% (ej 5%). De två testresultaten i kombination ger tvetydiga svar. Troligtvis är det covid (Transitory Change, TC) som spökar i KPSS.
    - Potentiell lösning för detta: Dummyvariabler. Dessa förhindrar modellen att dra iväg för mycket åt samma håll som outliers.
    - Alternativt kör Zivot-Andrews och låt testet hitta trendbrottet (Covid)

Gällande den otrimmade serien så behöver vi även där ta hänsyn till strukturbrottet.
    Samma två alternativ som för den trimmade: Dummyvariabel eller Zivot-Andrews.

In [ ]:
# Testar med Zivot-Andrews på otrimmade serien. Ska ta hänsy till trendbrott.
# ARCH-versionen tycks dock inte rapportera var brytpunkten är.
ZKPIF_f = ZivotAndrews(KPIF_full,
                       trend='ct',
                       method='bic',
                       max_lags=17)
print(ZKPIF_f.summary())

In [ ]:
ZKPIF_t = ZivotAndrews(KPIF_trimmed,
                       trend='c',
                       method='bic',
                       max_lags=12,
                       )
print(ZKPIF_t.summary())

In [ ]:
ZKPIF_f2 = zivot_andrews(KPIF_trimmed,
                         maxlag=17,
                         regression='c',
                         autolag="BIC")
ZKPIF_f2

In [ ]:
print(KPIF_trimmed.index[330])

Efter att ha testat både den trimmade och otrimmade serien med Zivot-Andrews så går det att konstatera att varken den otrimmade eller trimmade serien kan förkasta H0 när modellen endast har en konstant 'c'. Med konstant och trend 'ct' så kan vi däremot förkasta H0 på den trimmade serien, men inte på den otrimmade. 

Den samlade bedömningen efter dessa tester är att vi antar att den trimmade serien är stationär, men med en känd chock (covid-19) som på riktigt visar sig i andra halvan av 2021 där serien drar iväg från spannet den trendat kring sedan 1994.
    - KPSS förkastades på 10%, vilket troligtvis kan tillskrivas covid-19-chocken. Den samlade bedömningen från testerna och vad vi kan se i serien förändrar inte ståndpunkten kring att serien är stationär.

Vad gäller den otrimmade serien så kunde vi inte förkasta H0 i någon av testerna. I och med att det existerar flera regimer inom serien såsom 1980-1995 nedåttrendande inflation, 1995 introduktion av 2%-mål och slutligen covid-chocken, så blir serien problematisk att testa.
    - Även om det hade varit intressant att modella och prognosticera på hela serien så är risken att variansen exploderar och att värdet i prognoserna faktiskt försvinner.
        - Det är därav rimligt att lämna den otrimmade serien och fokusera på vad som faktiskt ser ut att gå att prognosticera på, den trimmade.
        Vi kan återkomma till den otrimmade senare och testa i känslighetsanalys.

- Det finns ett annat test, Phillip-Perron, som skulle kunna vara ett alternativ till ZA. Men det är sannolikt att resultat och insikt inte kommer skilja sig nämnvärt mot vad vi redan fått med ADF,ZA och KPSS.

Vi går vidare med att beräkna ACF och PACF, sedan plotta dem.

# ACF och PACF för trimmad serie

In [ ]:
# ACF och PACF för trimmad serie
# plot_acf/plot_pacf beräknar och plottar i samma anrop.
# ACF: hur många MA-termer (q). Tittar på var ACF skärs av (cut-off).
# PACF: hur många AR-termer (p). Tittar på var den PACF skärs av.
fig, axes = plt.subplots(2, 1, figsize=(12, 8)) # 2 diagram med en kolumn i storleken bredd = 12 höjd = 8
plot_acf(KPIF_trimmed['KPIF-12m-Change'], lags=40, ax=axes[0]) # axes anger vilken subplot grafen ritas
plot_pacf(KPIF_trimmed['KPIF-12m-Change'], lags=40, ax=axes[1])
plt.tight_layout()
plt.show()

Beräkningen av ACF och PACF görs med en funktion från statsmodels-biblioteket. plot_pacf och plot_acf är en kombinationsfunktion som beräknar och plottar. Eftersom bedömningar och beslut fattas på rent visuella grunder när det kommer till ACF och PACF passar dessa funktioner utmärkt.

Vi ser att den trimmade seriens ACF avtar gradvis mot 0 (når 0 ca 22 lags in). PACF ser vi däremot klippa mot 0 redan efter lag 1, varpå värdet vid lag 2 är inom 5%-konfidensintervall.

Vi har däremot något i PACF-grafen som sticker ut, och det är "spikes" som skjuter upp vid lag 13, 25 och 37. 
    
    - Varför sker detta?
        - Det troliga svaret skulle kunna vara chocken i serien som kom från Covid. Ett (i detta fall) falskt säsongsmönster upptäcks och tar då form av återkommande spikar i  PACF. Men eftersom vi vet att detta var ett engångsevent så kan vi förbise dessa spikar.
    
            * Engelska uttrycket för detta är Spurious seasonality.
    
            * Kan testas genom att rensa covid med dummyvariabel. Försvinner spikarna så var orsaken felaktig, om inte, så finns det troligtvis säsongsmönster.

Eftersom spikarna troligtvis orsakats av Covid så kan vi välja att förbise dem just nu.
    
    - Alternativt så kan vi redan i detta stadie rensa med dummyvariabel, skatta en OLS och plotta ACF, PACF på de skattade residualerna. Detta bör ge oss svar på om autokorrelationen kvarstår eller om den huvudsakligen försvinner när covid-effekten kontrolleras för.
        - Om spikarna reduceras kraftigt talar det för att de åtminstone delvis orsakades av covidrelaterade nivåförändringar snarare än av den underliggande stokastiska processen.
        - Jämför sedan med ACF/PACF på nivå-tidsserien.



In [ ]:
# Skapar covid-dummy från brytpunkt genererad i ZA-test och slutpunkt när serien återgått till 2-3% inflation (riksbankens mål).
# Ger: Från 2021-07-01 till 2023-12-01
covid_dummy = ((KPIF_trimmed.index >= '2021-07-01') & 
               (KPIF_trimmed.index <= '2023-12-01')).astype(int)

# Definierar dummyvariabel med statsmodel.
X = sm.add_constant(covid_dummy)

# Skattar OLS med dummyvariabel
model = sm.OLS(KPIF_trimmed['KPIF-12m-Change'], X).fit()

# sparar residualerna i KPIF_Clean
KPIF_Clean = model.resid

# Plottar ACF/PACF för residualerna
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(KPIF_Clean, lags=40, ax=axes[0], title='ACF - covid-rensad')
plot_pacf(KPIF_Clean, lags=40, ax=axes[1], title='PACF - covid-rensad')
plt.tight_layout()
plt.show()

Till skillnad från tidigare ACF så kan vi se att med covid-dummy så dör ACF ut redan vid lag 9. PACF klipper fortfarande vid lag 1, men de höga toppar som vi såg tidigare är betydligt mindre och har reducerats till enbart två. Det två kvarvarande topparna som knappt tar sig över 5% intervallet är troligtvis rester från det strukturella mönster i serien som kommer från att variablerna är rullande 12 mån förändring, dvs hänger ihop 12 månader.

I detta fall indikerar ACF/PACF på residualerna efter att covid-rensats att den sanna modellen troligtvis är av enklare form. Införandet av dummy-variabeln på covid är även motiverat givet resultaten.

    * Skillnad på ACF/PACF på nivåserien (KPIF_trimmed) och residualerna från OLS.
        
        - ACF och PACF kan beräknas både på den ursprungliga tidsserien och på residualer från en regression av e_t från skattad OLS, men de besvarar olika frågor. 
            - På Y_t: visar den totala autokorrelationsstrukturen i serien, 
              inklusive effekter från kända chocker som covid.
            - På OLS-residualer: visar autokorrelationsstrukturen *efter* att 
              ha rensat för en känd effekt (covid-dummy). Används explorativt för 
              att avgöra om spikar i ACF/PACF är äkta dynamik eller artefakter.
            
        - OLS-rensningen är explorativ – modellen skattas på originalserien 
          med covid-dummy som exogen variabel.
 
 Denna "detour" gjordes eftersom serien innehåller en stor tillfällig chock som, trots att den är förenlig med stationaritet, riskerade att förvränga ACF/PACF och eventuellt leda till felspecificering av modeller. På grund av detta undersökte vi chockens effekt på serien efter att ha sett tecken på potentiell inverkan i ACF/PACF på KPIF_trimmed.

 I det normala fallet, där serien redan kan betraktas som stationär och saknar tydliga deterministiska komponenter såsom trendbrott eller interventioner, analyseras ACF och PACF direkt på Y_t. Någon föregående OLS-rensning behövs normalt inte eftersom det inte finns någon uppenbar komponent att avlägsna innan korrelationsstrukturen studeras.

Nästa steg blir att skatta modeller på KPIF_trimmed med covid-dummies.

# Skatta modeller

In [ ]:
# Åtgärdar felmeddelanden som genererades av nedan (första) körning. Se markdown under för de varningar som genererades.

# Sätter frekvens på index.
KPIF_trimmed.index = pd.DatetimeIndex(KPIF_trimmed.index, freq='MS')



In [ ]:
# Eftersom ACF/PACF indikerar enklare modeller av slaget AR(1), AR(2), ARMA(1,1) och kanske ARMA(2,1)
# så börjar vi med att definiera dem.

ar1 = ARIMA(KPIF_trimmed, 
            order=(1,0,0), 
            exog=covid_dummy).fit(method_kwargs={'maxiter': 500})

ar2 = ARIMA(KPIF_trimmed, 
            order=(2,0,0), 
            exog=covid_dummy).fit(method_kwargs={'maxiter': 500})

arma11 = ARIMA(KPIF_trimmed, 
               order=(1,0,1), 
               exog=covid_dummy).fit(method_kwargs={'maxiter': 500})

arma21 = ARIMA(KPIF_trimmed, 
               order=(2,0,1), 
               exog=covid_dummy).fit(method_kwargs={'maxiter': 500})

# Adderade "method_kwargs = {'maxiter': 500}" på modellerna för att lösa problem med konvergering.
# Det betyder oftast att optimeringsalgoritmen helt enkelt behövde fler steg för att hitta optimum.

Efter initiala körningen av modellerna fick jag ett par varningar:
* ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
    - Detta fel är kopplat till att frekvensen på index i tidsserien inte är bestämt. I detta fall åtgärdas det genom att specificera månadsfrekvens. 

* ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to ")
    - Optimeraren (default är LBFGS) nådde max antal iterationer utan att hitta ett stabilt minimum.
    Potentiell lösning är maxiter=500 so ger fler försök, och method='powell' byter optimeringsalgoritm. Powell är ibland bättre för problem där gradienten är svårberäknad.

* UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters')
    - ARIMA-skattningen startar från initiala gissvärden för parametrarna innan den optimerar. Denna varning får man om startvärdena råkar motsvara en icke-stationär process (AR-rötter utanför enhetscirkeln).
    Lösningen bör vara att ange egna startvärden som modellen börjar optimera ifrån. 

* UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.')
    - Samma problem fast för MA-delen. Startvärdena råkar hamna utanför invertibilitetsvillkoret. Åtgärden är även här att speca initiala startvärden som ger invertibel start.

Påverkar start_params slutresultat? Ja, det kan påverka slutresultatet om optimeringslandskapet har flera lokala minima. En bättre startpunkt kan hitta ett bättre (lägre) minimum. Men i välspecificerade modeller bör olika startvärden konvergera till samma globala minimum. Skillnaden är mest i om modellen konvergerar överhuvudtaget.

konvergensvarningar på komplexa modeller (AR(2), ARMA) men inte på AR(1) indikerar att data stödjer en enklare modellstruktur. Detta är i linje med ACF/PACF.

In [ ]:
# Jämför modeller med AIC och BIC. 
# Lägre värde indikerar bättre modell.
for name, m in [('AR(1)', ar1), ('AR(2)', ar2), ('ARMA(1,1)', arma11), ('ARMA(2,1)', arma21)]:
    print(f'{name}: AIC={m.aic:.2f}, BIC={m.bic:.2f}')

Konvergensvarning uppstod initialt för AR(2), ARMA(1,1) och ARMA(2,1), men dessa löstes till slut.

Hur tolkar man AIC och BIC-resultat?
    - AIC och BIC är poäng-metoder som hjälper en att välja mellan regressionsmodeller baserat på hur bra modellen passar datan. Men i och med att modeller med fler adderade variabler passar datan bättre generellt så straffar AIC/BIC modeller med fler variabler. Den bästa modellen är helt enkelt den som kan förklara "mest" med så få variabler som möjligt. 
        - AIC är däremot lite "snällare" och straffar inte ytterligare variabler lika hårt som BIC. Så BIC kommer generellt sett att premiera mer "parsimonious" modeller med färre variabler.

        - "Raftery (1995) says that a BIC difference of 10 equals a Bayes factor of 150. Basically, a difference of 10 is strong evidence that the model with the lowest bic fits best. Raftery, A. E. (1995). Bayesian Model Selection in Social Research. Sociological Methodology, 25, 111-163."

I resultatet ovan kan vi se att AR(1)-modellen, enligt BIC, är den som passar serien bäst (-3213,78 vs -3210,2) med en diff på -3,58. Men baserat på AIC så skulle AR(2) vara den bästa modellen.
    - Skillnaden i AIC är däremot endast -0,38 i kontrast till de hela -3,58 i BIC. 

Resultaten pekar därav på att AR(1) är den modell vi  bör använda, vilket även är i linje med vad vi såg i PACF där den klippte till ~0 efter lag 1.


In [ ]:
# Ljung-Box 
# H0: Residualerna är oberoende
# H1: Residualerna är inte oberoende och uppvisar seriekorrelation/autokorrelation

# Vi vill inte att H0 ska förkastas. Att förkasta H0 betyder att modellen saknar parametrar som förklarar variansen.
# Vid förkastande av H0 behöver modellen troligtvis omspecificeras (till en mer komplex modell).

residual = ar1.resid

lb = acorr_ljungbox(residual, lags=[6, 12, 24], return_df=True)
print(round(lb, 6))


In [ ]:
# Jarque-Bera
# H0: Datan (residualerna) är normalfördelade
# H1: Datan (residualerna) är inte normalfördelade 

# Att förkasta H0 tyder på skevhet eller excesskurtosis i datan. Detta är ett problem för ARMA specifikt på grund av
# att konfidensintervall kräver symmetri för att tolkas korrekt. Parametrarna är däremot fortfarande konsistenta.

jb = jarque_bera(residual)
print(f'JB-Stat: {jb[0]:2f}, P-värde: {jb[1]:4f}')

Ljung-Box på residualerna av AR(1)-modellen förkastade H0 vid alla specificerade lags. Inte vad vi hade hoppats på. Men detta beror troligtvis på att det finns dynamik i serien som modellen inte lyckas fånga och justeringar behöver därmed göras.
    - De kvarvarande spikarna i PACF (även efter införandet av Covid-dummy) hintade om att det troligtvis fanns kvarvarande autokorrelation. En del misstänktes kunna härledas till seriens struktur (rullande 12 månader), men det är möjligt att det finns annan dynamik som spelar in här också.
    - ACF/PACF gav stöd för AR(1)-modell, men det är möjligt att vi behöver addera säsongsvariabel och ytterligare exogen variabel.

Förkastningen av H0 i Jarque-Bera testet tyder på icke-normalfördelade residualer. För en vanli AR(1) är detta ok då parametrar fortfarande är konsistenta och unbiased. Test är även ensidiga här (till skillnad från ARMA), men prognosintervallen kommer däremot att påverkas av icke-normalitet

För att diagnosticera detta så plottar vi ACF/PACF på residualerna.

In [ ]:
# Plottar ACF/PACF
# Statsmodels.graphics plot_acf/pacf

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(residual, lags=40, ax=axes[0], title='ACF - AR(1) residualer')
plot_pacf(residual, lags=40, ax=axes[1], title='PACF - AR(1) residualer')
plt.tight_layout()
plt.show()

Varför skiljer sig ACF/PACF på residualerna från AR(1) och de från OLS?
    - OLS med covid-dummy tar bort nivåskiftet (covid-chocken) vilket lämnar kvar den "normala" inflationsdynamiken. Men i AR(1) som modellerar ett steg tillbaka i tiden missar 12-månadersdynamiken, vilket då hamnar i residualerna.

Som vi ser ovan så får vi negativa spikes i ACF/PACF vid lag 12 som då hintar om att seriens mönster fångas upp i residualerna (vilket vi inte vill).
    - För att lösa detta behöver vi troligtvis utöka modellen. Men innan vi adderar en eventuell MA-term så kan vi testa SARIMAX istället för enbart ARIMAX.

In [ ]:
# Kikar ar1-resultat om covid-dummyn är signifikant
# Ingen större vikt bör i detta fall läggas vid LB och JB test som presenteras för lag 1 när vi är intresserade av 
# främst de senare lagsen (som tidigare undersökts).
print(ar1.summary())

I och med att covid-dummyn är signifikant så kan vi anta att mycket av covid-effekten fångas upp där. De spikar vi ser i ACF/PACF är därav troligtvis hänförliga till det strukturella 12mån mönstret och möjligtvis andra faktorer (ekonomisk dynamik).

Men den 12mån rullande strukturen eliminerar säsongseffekter under året, så vad är det då som återstår?
    - Svag äkta säsongseffekt?
    - AR(1) fångar alltså hur yt beror på yt-1 men inte yt och yt-12 (samma månad förra året). Månaden delar alltså 11 datapunkter med sina nästliggande datapunkter men inte några med samma månad föregående år. Denna brytpunkt i korrelationsstrukturen skapar spiken vid lag 12 när datapunkten vid tid t inte har något gemensamt med lag 12.

Vi skattar därav en SARIMAX specad med 12 lags för att se om vi kan få kontroll över detta problem.

# Skatta SARIMAX och utvärdera

In [ ]:
# Skattar SARIMAX - AR(1) med säsong och exogen variabel.

sarx1 = SARIMAX(KPIF_trimmed, 
                order=(1,0,0),
                seasonal_order=(0,0,1,12),
                exog=covid_dummy).fit(disp=False, maxiter=200)
# Löser konvergensproblem med maxiter

print(sarx1.summary())

# "disp=False" anges för att tysta output under skattning (iterationer, konvergensinformation), rent kosmetiskt.
# "Seasonal_order=(P,D,Q,S)" följer samma logik som order för modellen men behöver inte nödvändigtvis spegla order 
# av den vanliga modellen. Dvs vi kan ha order=(1,0,0) och seasonal_order=(0,0,1,12) men sällan motiverat. Vanligare
# är att ha av samma order eller under (exempel: order=(2,0,0) och seasonal_order=(1,0,0,12)).

In [ ]:
# Ljung-Box på SARIMAX-modellen för lags 6, 12 och 25
lb_sarx1 = acorr_ljungbox(sarx1.resid, lags=[6,12,24], return_df=True)
print(lb_sarx1)

In [ ]:
# Plottar ACF/PACF för SARIMAX-modellen
# Statsmodels.graphics plot_acf/pacf

resid_sarx1 = sarx1.resid

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(resid_sarx1, lags=40, ax=axes[0], title='ACF - AR(1) residualer')
plot_pacf(resid_sarx1, lags=40, ax=axes[1], title='PACF - AR(1) residualer')
plt.tight_layout()
plt.show()

Den skattade SARIMAX-modellen ger sken av att vara en uppgradering från den enklare ARIMAX-modellen. BIC signallerar stor förbättring, den nya termen är även signifikant och spikarna i ACF/PACF-lags är även mindre dramatiska än tidigare.

Ljung-Box förkastas däremot fortfarande vilket inte är önskvärt. Att addera den där MA-termen är troligtvis nödvändigt och skulle kanske kunna ge bättre resultat (motiverat av både PACF (negativa spikar) och förkastat LB-test).

Men, istället för att skatta enstaka modeller för sig så kan vi använda oss av en funktion, "auto_arima", som söker igenom flera kombinationer och returnerar den bäst passande modellen enligt ett valt informationskriterium.
    - En begränsning är att auto_arima optimerar statistiskt utan förståelse för serien. Den känner inte till eventuella strukturella brott, konstruktionsegenskaper eller liknande som kan behöva finjusteras för. Resultatet bör därför tolkas tillsammans med övrig diagnostik och kännedom kring serien.

# Skatta med Auto_Arima

In [ ]:
# Specar vad auto_arima ska köra igenom för typ av modeller.
# Intercept = true pga 2%-målet
# Seasonal = true pga 12m-mönster

auto_model = auto_arima(KPIF_trimmed,
                        X=covid_dummy.reshape(-1,1),
                        start_p=0, max_p=3,
                        start_q=0, max_q=3,
                        start_P=0, max_P=2,
                        start_Q=0, max_Q=2,
                        m=12,
                        D=0,
                        d=0,
                        with_intercept=True,
                        seasonal=True,
                        information_criterion='bic',
                        stepwise=True,
                        trace=True)
print(auto_model.summary())

Initiala körningen presenterade en SARIMAX(1, 0, 0)x(0, 0, [1], 12) utan intercept. Även om modellen är en bra vägvisare så finns det en del problematik kopplat till den presenterade modellen.
    - Utan intercept antas långsiktiga prognoser konvergera mot 0. Går emot riksbankens 2%-mål.
    -  AR-koefficienten är nästan 1 med konfidensintervall [0.9994 till 1.003]

Första problemet med intercept kan vi åtgärda genom att tvinga sökningen inkludera intercept "with_intercept=True", alternativt addera ett manuellt.

Andra problemet har att göra med "parameter redundans" eller "nära icke-inverterbarhet" där AR-koeficienten har ett värde med konfidensintervall runt 1. 
    - Detta har troligtvis att göra med den rullande 12m-strukturen som orsakar persistens.

    Problem som skulle behöva åtgärdas
1. LB förkastar
2. AR-term nära 1 i många modeller
3. Heteroskedasticitet
4. JB förkastar

Men kan allt lösas?

George E. P. Box: "All models are wrong, but some are useful." från Robustness in Statistics. Poängen är att modellval i slutändan avgörs av användbarhet, inte perfektion.

Clive W. J. Granger och Paul Newbold diskuterar redan i klassiska prognosböcker att diagnostik är viktig men att prognosjämförelser out-of-sample är det slutliga testet.

Rob J. Hyndman och George Athanasopoulos betonar i Forecasting: Principles and Practice att modeller ska bedömas på prognosfel på data som inte användes vid skattningen.

James D. Hamilton påpekar i Time Series Analysis att makroserier ofta uppvisar strukturbrott, heteroskedasticitet och andra avvikelser som gör att diagnostiken sällan ser "perfekt" ut.

Det betyder inte att man ska ignorera Ljung–Box. Men om du har två modeller där:

Modell  Ljung-Box	RMSE
A	    Perfekt 	1.2
B	    Förkastar	0.7
så väljer man normalt B för prognoser.

    Väg framåt

* Accepterar att datan inte kommer att klara alla tester. Det kommer finnas risk att intervall blir mindre tillförlitliga.
* Prognosticera och utvärdera out-of-sample-prestanda på den 12m rullande serien.
* Konstruera ny serie med månadsförändring i inflation (bör lösa strukturproblemet med autokorrelation i 12 lags) och kör samma spår som initiala serien. 

Modeller att utvärdera med pseudo out-of-sample-prognoser:
* AR(1) + covid-dummy
* SARIMAX(1,0,0)(0,0,1,[12]) + covid-dummy

Benchmarka mot mean och naive-prognos

Mean-"prognos": Detta är det historiska medelvärdet som vi jämför prognoser mot. Är historiskt medelvärde bättre predikator för faktiskt utfall i de flesta scenarios? 
    - Gissar att nästa observation blir lika med historiskt medelvärde.

Naive-prognos: Denna "prognos" antar att nästa observation blir samma som den tidigare observationen. 
    - Y_t+h|t = Y_t
    - Varför? Många makroekonomiska serier är väldigt (extremt?) persistenta och gör att en bra gissning för vad nästa periods observation blir är exakt vad observationen är idag.
        - Låter dumt, men förvånansvärt svårslaget. Används därav ofta som benchmark vid prognosticering.
        - Fungerar som en tröskel för acceptabel modell. Kan vi inte slå naive, varför använda modellen om det knappt finns prognosvärde?

# Pseudo out-of-sample och prognosutvärdering

In [ ]:
# Prognosloop (Expanding Window Pseudo out-of-sample forecast + evaluation)
# Prognoser enbart för perioder vi har "facit" på -> Utvärderar modellprestation.
# Utvärderar AR(1) och SARIMAX som valdes tidigare. Benchmarkas mot historiskt medelvärde och Naive


forecast_horizons = [1, 3, 6, 12]   # Definierar forecast_horizon för antalet steg framåt vi prognosticerar.
# train_end = '2023-12-01'            # Anger datum/index för sista träningsdatan, dvs modellen skattas på datan fram till och med 2023-12-01
# results = {}                        # Sparar resultat i "results" (dictionary)  

for train_end in ['2023-01-01', '2023-06-01', '2023-12-01']: # upd (additional train periods) for comparison in the end
    results = {}
    for h in forecast_horizons: # Definierar yttre for-loop för prognoshorisonterna. Första körs h=1, sedan h=3 osv. 4 separata prognosutvärderingar görs.

        # Skapar tomma listor
        actuals = []            # Fylls med faktiska utfall
        preds_sarimax = []      # Fylls med SARIMAX-prognoser. Respektive h-step ahead som motsvarar forecast_horizon sparas i listan . 
        preds_ar1 = []          # AR(1)-prognoser
        preds_naive = []        # Naive-prognoser
        preds_mean = []         # Historiska medelvärdet. Medelvärdet på träningsdatan används som prognosvärde.

        # Räknar hur ofta SARIMAX och AR(1) inte konvergerar. Eftersom vi hade koefficienter nära 1 är detta intressant.
        failed_sarimax = 0
        failed_ar1 = 0

        evaluation_length = len(KPIF_trimmed.loc[train_end:]) - h + 1 # Hur många prognoser som ska göras (antal möjliga "forecast origins").
                                                                  # .loc[train_end:] specificerar datan från train_end och allt efter ":".
                                                                  # Ser till att faktiska utfall finns tillgängliga för hela prognosperioden.

        for t in range(evaluation_length):  # Expanding window - for loop som flyttar prognosen framåt varje steg. Startar från t = 0
                                            # Första iteration körs på data fram till train_end. Andra iteration körs på data fram till train_end + 1 osv. 

            train = KPIF_trimmed.iloc[:len(KPIF_trimmed.loc[:train_end]) + t] # För varje steg växer datamängden med + t.
            cd = covid_dummy[:len(train)]  # Definierar covid-dummyn så att den matchar träningsdatan längdmässigt.
            
            # Skapar framtida värden för covid-dummyn och Ser till att de tar värdet 0 efter covid-perioden i out-of-sample.
            raw_future = covid_dummy[len(train):len(train)+h]
            future_exog = np.zeros(h)
            future_exog[:len(raw_future)] = raw_future
            future_exog = future_exog.reshape(-1, 1) # Matchar array till SARIMAX och AR(1)

            # Skattar SARIMAX utefter den modell som enligt AIC/BIC var bäst.
            model = SARIMAX(train,
                            order=(1, 0, 0),
                            seasonal_order=(0, 0, 1, 12),
                            exog=cd,
                            trend='c').fit(disp=False, maxiter=200)

            if not model.mle_retvals['converged']: # If-funktion som räknar körningar där optimering misslyckades.
                failed_sarimax += 1

            fc = model.forecast(steps=h, exog=future_exog) # Prognosen för SARIMAX

            # Skattar AR(1) baserat på vad vi sett i datan (ACF/PACF).
            ar1 = ARIMA(train,
                        order=(1, 0, 0),
                        exog=cd).fit(method_kwargs={'maxiter': 200})

            if not ar1.mle_retvals['converged']: # Samma som för SARIMAX
                failed_ar1 += 1

            fc_ar1 = ar1.forecast(steps=h, exog=future_exog) # Prognosen för AR(1)

            actual_idx = len(train) + h - 1 # Faktiskt utfall som ska jämföras mot. Specas som rad

            # Förhindra att jämföra mot data som inte finns. Sparar utfall så länge jämförelseindexet är inom serien. 
            if actual_idx < len(KPIF_trimmed):
                preds_sarimax.append(fc.iloc[-1])                   # SARIMAX-prognoser. Sparar alltid sista värdet i matrisen [-1] (h= 1, 3, 6 eller 12), ej de mellanliggande prognoserna.
                preds_ar1.append(fc_ar1.iloc[-1])                   # AR(1)-Prognoser
                preds_naive.append(train.iloc[-1, 0])               # Benchmark. 
                preds_mean.append(train.iloc[:, 0].mean())          # Benchmark. Prognos sätts till medelvärdet. [:,0] = alla rader, första kolumnen .(mean) ger sedan medelvärdet
                actuals.append(KPIF_trimmed.iloc[actual_idx, 0])    # Faktiska utfall som hämtas från specifik rad/datum i tidsserien

        # Skapa numpy-arrayer för modeller + faktiska utfall. 
        # Används för att beräkna bl.a. RMSE och MAE så att vi kan utvärdera.
        a = np.array(actuals)
        s = np.array(preds_sarimax)
        r = np.array(preds_ar1)

        results[f'h={h}'] = {
            'SARIMAX_RMSE': np.sqrt(np.mean((a - s)**2)),
            'SARIMAX_MAE':  np.mean(np.abs(a - s)),
            'AR1_RMSE':     np.sqrt(np.mean((a - r)**2)),
            'AR1_MAE':      np.mean(np.abs(a - r)),
            'NAIVE_RMSE':   np.sqrt(np.mean((a - np.array(preds_naive))**2)),
            'NAIVE_MAE':    np.mean(np.abs(a - np.array(preds_naive))),
            'MEAN_RMSE':    np.sqrt(np.mean((a - np.array(preds_mean))**2)),
            'MEAN_MAE':     np.mean(np.abs(a - np.array(preds_mean))),
            'e_sarimax':    a - s, # Prognosfel SARIMAX (faktiskt - prognos)
            'e_ar1':        a - r  # Prognosfel AR(1) (faktiskt - prognos)
        }

    print(pd.DataFrame(results).T.drop(columns=['e_sarimax', 'e_ar1'])) # Skapar en tabell av resultaten "results". (Prognosfelen är exkluderade)

In [ ]:
# Diebold-Mariano (DM) och Harvey-Leybourne-Newbold (HLN) test
# Testar om skillnaden i prognosfel mellan SARIMAX och AR(1) är statistiskt signifikant.
# H0: Modellerna har samma prognosförmåga. Förlustdiff = 0
# H1: Modellerna har olika prognosförmåga.
# DM använder normalfördelning och är asymptotiskt korrekt vid stora stickprov.
# HLN är en small-sample-korrigering av DM som använder t-fördelning.
# Positivt DM/HLN innebär att SARIMAX har högre förlust än AR(1), dvs AR(1) är bättre

print("Diebold-Mariano & HLN-test: SARIMAX vs AR(1)")
print("-" * 45)

for h in forecast_horizons:
    e_s = results[f'h={h}']['e_sarimax']
    e_r = results[f'h={h}']['e_ar1']

    d = e_s**2 - e_r**2
    T = len(d)
    
    # DM
    dm_stat = d.mean() / (np.std(d, ddof=1) / np.sqrt(len(d)))
    p_dm = 2 * (1 - stats.norm.cdf(np.abs(dm_stat)))
    
    # HLN
    hln_stat = dm_stat * np.sqrt((T + 1 - 2*h + h*(h-1)/T) / T)
    p_hln = 2 * (1 - stats.t.cdf(np.abs(hln_stat), df=T-1))

    stars_dm  = "***" if p_dm   < 0.01 else "**" if p_dm   < 0.05 else "*" if p_dm   < 0.10 else ""
    stars_hln = "***" if p_hln  < 0.01 else "**" if p_hln  < 0.05 else "*" if p_hln  < 0.10 else ""
    
    print(f"h={h:2d} | DM={dm_stat:+.3f} p={p_dm:.3f}{stars_dm:3s} | HLN={hln_stat:+.3f} p={p_hln:.3f}{stars_hln:3s}")

     Utvärdering av (pseudo out of sample) prognoser från KPIF_trimmed 12m rullande inflationsdata

Trots motgångarna vid modellvalet var målet att producera så träffsäkra och robusta prognoser som möjligt. De två modeller som valdes för prognosutvärdering var AR(1), som motiverades tidigt i analysen, och en SARIMAX(1,0,0)x(0,0,1,12) som i AIC/BIC mått mätta tycktes framstå som den bäst lämpade modellen att hantera den strukturella 12månaders dynamiken serien hade.
     Problemen som vi tampades med. Vilken betydelse har de för prognoser?

          1. LB förkastar H0. Residualer innehåller fortfarande autokorrelation.
          2. AR-term nära 1 i flera modeller.
          3. Heteroskedasticitet indikeras.
          4. JB förkastar normalitetsantagande.
     
     För prognoser är detta inte nödvändigtvis något som skapar några större problem. Förkastandet av Ljung-Box, som kan anses vara det mest problematiska, betyder att residualerna fortfarande innehåller systematisk information och att modellen inte lyckats fånga all dynamik. Detta måste inte snedvrida prognoserna, men tyder på att det finns utrymme för en mer optimal modell.
     Att AR-termen tar ett värde mycket nära 1.0 är ett tecken på hög persistens, vilket kanske inte är så konstigt när närliggande observationer delar 11 gemensamma datapunkter (12-månadersförändring). På kort sikt är det oftast inget negativt och kan till och med vara en fördel eftersom förändring ofta sker långsamt i normala fall. Men på lång sikt ackumuleras eventuella skattningsfel, och eftersom de hänger kvar länge kan detta leda till bredare konfidensintervall som upplevs mindre precisa än önskat. Den observerade heteroskedasticiteten och förkastandet av Jarque-Bera-testet antyder också en sådan osäkerhet i modellen, just vid längre prognoshorisonter.
     Trots dessa problem så påverkas punktprognoserna troligtvis minimalt på grund av seriens struktur. Men det är viktigt att komma ihåg begränsingarna i konfidensintervallets precision och att modellerna troligtvis underskattar sannolikheten för stora utfall när slutsatser ska formas.
          - Förmåga att prognosticera regimskiften eller black-swan-events för inflation är däremot extremt svårt eller näst intill omöjligt. Anledningen är att prognoserna förutsätter att framtiden följer ungefär samma statistiska process som den historiska datan. 

För att utvärdera de två modellerna så kör vi en expanding-window pseudo out-of-sample prognosutvärdering på datan. Där modellerna skattas på data från start och fram till och med 2023-12-01. Fyra h-stegsprognoser ( 1, 3, 6 och 12 steg framåt) testas och jämförs mot det faktiska utfallet för att se om modellerna hade klarat av att prognosticera perioder med någorlunda precision. Två benchmarks (Naive och Mean) adderas även till utvärderingen i syfte att kunna avgöra om modellen faktiskt har någon form av prediktiv kapacitet.
RMSE (Root Mean Squared Error) och MAE (Mean Absolute Error) för varje modell och benchmark beräknas och används sedan för att avgöra vilken modell som presterade bäst. Därefter gör vi ett DM och HLN-test för att avgöra om skillnader i prognoskapacitet är signifikanta. 

Resultat från utvärdering
________________________________________________________________________________________________
     SARIMAX_RMSE SARIMAX_MAE | AR1_RMSE   AR1_MAE  | NAIVE_RMSE NAIVE_MAE | MEAN_RMSE  MEAN_MAE
h=1      0.004837    0.003678 |  0.004765  0.003748 |  0.004696  0.003749  | 0.006527  0.005345 
h=3      0.008938    0.007138 |  0.006491  0.005488 |  0.00671   0.005675  | 0.006167  0.005046  
h=6      0.014249    0.012116 |  0.009096  0.008166 |  0.009958  0.008833  | 0.00646   0.005308  
h=12     0.026506    0.025238 |  0.008889  0.006807 |  0.010385  0.008283  | 0.00671   0.005391   

Diebold-Mariano & HLN-test: SARIMAX vs AR(1)
---------------------------------------------
h= 1 | DM=+0.074 p=0.941    | HLN=+0.072 p=0.943   
h= 3 | DM=+1.798 p=0.072*   | HLN=+1.631 p=0.115   
h= 6 | DM=+2.461 p=0.014**  | HLN=+1.896 p=0.071*  
h=12 | DM=+8.123 p=0.000*** | HLN=+2.925 p=0.009***
________________________________________________________________________________________________

Med RMSE, MAE och DM-testet uträknat är utvärderingen inte särskilt komplicerad. Lägst RMSE och MAE är den modell som prognosticerat bäst.
För prognoser h=1-steg framåt kan vi se en relativt jämn prestation mellan SARIMAX, AR(1) och Naive-prognoserna. SARIMAX med lägst MAE (0,003678) (MAE straffar stora fel mindre än RMSE), Naive med lägst RMSE (0,004696) och AR(1) strax över med RMSE=0.004765. Detta kan sannolikt förklaras av modellernas struktur. Naive som antar senaste observationens värde har ofta hög träffsäkerhet på kort sikt då större förändringar i inflation sällan sker under korta tidshorisonter, och SARIMAX samt AR(1) som hade en AR-koefficient mycket nära 1 kommer då anta väldigt liten förändring h=1 steg framåt från den senaste observationen. Men efter ett steg framåt så händer något intressant, som troligtvis beror på det som var en styrka för h=1 steg framåt. Både SARIMAX och Naive tappar prognosprecision efter h=1. Istället är det AR(1) och historiska medelvärdet som bäst tycks prognosticera utfall h=3+ steg efter datan som modellerna skattades på.
     - Den troliga förklaringen till SARIMAX försämrade prestation är att säsongs-MA(12) succesivt förlorar betydelse längre ut och ersätts med sitt väntevärde. Tillsammans med en AR-koefficient nära 1 så riskerar skattningsfelen fortplantas och ackumuleras över tid.

Av de två skattade modellerna indikerar Diebold-Mariano-testet att de skillnader vi ser efter h=1 steg framåt mellan SARIMAX och AR(1) är signifikanta. HLN som är mer konservativt indikerar däremot att skillnaderna är signifikanta först vid h=6 steg framåt. SARIMAX RMSE och MAE på h>=3 indikerar ett drastiskt tapp i prediktiv precision med en kraftig försämring vid h=12 (RMSE=0.026506, MAE=0.025238) medan AR(1) ( RMSE=0.008889, MAE=0.006807) tycks prestera helt ok även på längre tidshorisonter. Trots denna starka prestation jämfört med SARIMAX så lämnas något att önska när man jämför mot det historiska medelvärdet.

Mean-"prognosen" (historiska medelvärdet) är svårt att slå på längre horisonter i serier som dessa. Troligtvis på grund av att persistensen i 12m-serien gör att dynamiska modeller endast ger begränsad ytterligare information. Att serien är stationär samt att den under utvärderingsperioden fluktuerar kring en relativt stabil nivå bidrar även till att det just detta mått är ovanligt starkt som benchmark.
     - Det är möjligt att MoM som serie har en renare stokastisk struktur då närliggande datapunkter inte delar ~ 90% av strukturen med varandra. MoM bör på så vis ge modellerna bättre förutsättningar att fånga faktiskt inflationsdynamik, vilket vi senare ska undersöka.

# Prognos 12m rullande data - start 2026-05-01 - AR(1)

In [ ]:
# Prognosticera inflation med AR(1) från seriens slut 2026-05-01
# Separat skattning från AR(1)-modellen som utvärderades ovan.

# Skattar AR(1)-modell
ar1fc_model = ARIMA(
    KPIF_trimmed, # Baseras på all tillgänglig data, till och med 2026-05-01
    order=(1,0,0),
    exog=covid_dummy # Tar endast värde 1 under inflationschocken från covid-perioden, 2021-07-01 till 2023-12-01
).fit()

# Definiera exogen variabel (covid_dummy) för prognoser
future_exog = np.zeros((12,1))  # Matris med 12 rader och 1 kolumn (fylld med 0:or)
                                # Varför 12,1? Måste matcha matriser. 12 rader, en för varje steg och kolumner för att matcha antalet exogena variabler (en i detta fall).

# Definiera prognos. Antal steg h framåt och exogen variabel
fc = ar1fc_model.get_forecast(  # ".get_forecast" vs "forecast": .forecast ger endast prognoser. .get_forecast ger ett forecast-objekt som innehåller bl.a. prognoser, standardfel och konfidensintervaller, vilket är mer intressant när man prognosticerar framtid.
    steps=12,
    exog=future_exog # Tar värdet 0 efter 2023-12-01, så även 0 i prognosen 2026-05-01 ->
)

forecast = fc.predicted_mean    # Hämtar prognosbanan. Eftersom modellen genererar sannolikhetsfördelning så hämtar "predicted_mean" E[Y_t+h​] (förväntat värde).
ci = fc.conf_int()              # Hämtar konfidensintervallet (lower och upper bound) 95%-intervall.

# Printar de specifika +h prognoser av intresse.    
print("1 månad:", forecast.iloc[0])     # Printar första värdet i prognosserien
print("3 månader:", forecast.iloc[2])   # Printar 3e värdet
print("6 månader:", forecast.iloc[5])   # 6e värdet
print("12 månader:", forecast.iloc[-1]) # 12e/sista värdet.

print("\nHela prognosbanan:")
print(forecast)

print("\n95%-konfidensintervall:")
print(ci)

Modellen tränas på hela seriens data och prognosticerar sedan 12 stycken YoY inflationsförändringar med start 2026-06-01 till och med 2027-05-01. AR(1)-modellen som används producerar en prognosbana där mean ligger runt 1,8% för de 12 prognoserna, vilket känns rimligt i och med riksbankens 2%-mål och historiska problem att faktiskt underskjuta det innan covid. Vad som däremot bör belysas är den ökade osäkerheten i prognoserna som växer för varje steg ut från modellens sista träningsdatum. Upper och lower, "spreaden", eller konfidensintervallet på prognosen börjar med ett lower-bound = 1,05% och upper-bound = 2,5225%, vilket redan är ganska brett. Detta intervall växer sig däremot allt större redan efter h=2 steg ut från start där prognosintervallet redan efter h=6 steg ut är 0,1338% till 3,4928%, alltså extremt brett sett till vad som är rimliga förväntningar.

    - Det breda konfidensintervallet är till tor del en effekt av att AR-koefficienten som skattas på 12m rullande är väldigt nära 1. Skattningen gör att prognosvariansen växer näst intill linjärt med prognoshorisonten. Detta förstärks även av den identifierade heteroskedasticiteten och kvarvarande autokorrelationen i residualerna, vilket betyder att vi bör tolka intervallen med stor försiktighet.

# Analys 2: MoM KPIF-data

In [ ]:
# Skapar MoM-förändring från KPIF-datan.
inf_data['KPIF-MoM'] = inf_data['KPIF'].pct_change(periods=1)

# Isolerar till egen dataframe.
KPIF_MoM = inf_data[['period','KPIF-MoM']].copy().dropna().reset_index(drop=True)

# Sätter datum som index.
KPIF_MoM = KPIF_MoM.set_index('period')

# Trimmar serien likt KPIF_Trimmed.
KPIF_MoM_trimmed = KPIF_MoM.loc['1994-01-01':'2026-05-01']

# Anger frekvens
KPIF_MoM_trimmed.index = pd.DatetimeIndex(KPIF_MoM_trimmed.index, freq='MS')

In [ ]:
# Fördelning och medelvärde
KPIF_MoM_trimmed.describe()

In [ ]:
# Plottar grafer och granskar.
px.line(KPIF_MoM_trimmed, 
        y='KPIF-MoM').show()

# ADF, KPSS, ACF/PACF

In [ ]:
# ADF-test
kpifmom_adf = ADF(KPIF_MoM_trimmed['KPIF-MoM'],
                  trend='c',
                  method='bic',
                  max_lags=18)
print(kpifmom_adf.summary())

In [ ]:
# KPSS-test
kpifmom_kpss = KPSS(KPIF_MoM_trimmed['KPIF-MoM'],
                    lags=12,
                    trend='c')
print(kpifmom_kpss.summary())

In [ ]:
# ACF och PACF
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(KPIF_MoM_trimmed['KPIF-MoM'], lags=40, ax=axes[0])
plot_pacf(KPIF_MoM_trimmed['KPIF-MoM'], lags=40, ax=axes[1])

plt.tight_layout()
plt.show()

ACF och PACF klipper direkt till 0. Flertal signifikanta spikes som tyder på säsongsvariation i serien. Tydlig 12-månaders effekt, men även en form av halvårsvis effekt kan utläsas. I och med att serien beter sig väldit likt vitt brus (white noise) så har vi inte så mycket visuellt att gå på i modellskattningen. Det blir därav rimligt att direkt skatta modeller med auto_arima och sedan basera valet på de presenterade alternativen. 

# Skatta modeller med auto_arima

In [ ]:
# Auto_arima för MoM-serien
MoM_auto_arima = auto_arima(KPIF_MoM_trimmed,
                            start_p=0, max_p=3,    
                            start_q=0, max_q=3,
                            start_P=0, max_P=3,
                            start_Q=0, max_Q=3,
                            m=12,
                            d=0, D=0,
                            trend='c',
                            seasonal=True,
                            information_criterion='bic',
                            stepwise=True,
                            trace=True
                            )
print(MoM_auto_arima.summary())

Auto_arima testar flertalet modeller och en del kandidater tycks prestera väldigt lika sett till BIC. 

ARIMA(0,0,0)(1,0,1)[12]             : BIC=-3339.853, Time=0.52 sec (*)
ARIMA(0,0,0)(2,0,2)[12] intercept   : BIC=-3339.745, Time=0.94 sec 
ARIMA(0,0,0)(1,0,1)[12] intercept   : BIC=-3339.853, Time=0.52 sec (*)
ARIMA(0,0,0)(1,0,2)[12] intercept   : BIC=-3338.691, Time=0.76 sec
ARIMA(1,0,0)(1,0,2)[12] intercept   : BIC=-3337.064, Time=1.93 sec (*)
ARIMA(1,0,0)(2,0,2)[12] intercept   : BIC=-3330.143, Time=1.03 sec

Utifrån dessa väljer jag de tre stjärnmarkerade och skattar dem separat. Jag testar även om covid-dummyn som vi använde i den föregående serien är signifikant för denna med. Förhoppningen är att de problem vi uppmärksammade i föregående tidsserien inte dyker upp i samma omfattning här.

In [ ]:
model_1 = SARIMAX(KPIF_MoM_trimmed,
                  order=(0,0,0),
                  seasonal_order=(1,0,1,12),
                  exog=covid_dummy).fit(disp=True, maxiter=200)


model_2 = SARIMAX(KPIF_MoM_trimmed,
                  order=(0,0,0),
                  seasonal_order=(2,0,2,12),
                  exog=covid_dummy,
                  trend='c').fit(disp=True, maxiter=200)

model_3 = SARIMAX(KPIF_MoM_trimmed,
                  order=(1,0,0),
                  seasonal_order=(1,0,2,12),
                  exog=covid_dummy,
                  trend='c').fit(disp=True, maxiter=200)

for name, m in [('(0,0,0)x(1,0,1)[12]', model_1),('(0,0,0)(2,0,2)[12]', model_2),('(1,0,0)(1,0,2)[12]', model_3)]:
    print(f'{name}: BIC:{m.bic:.2f}')
    lb = acorr_ljungbox(m.resid, lags=[6, 12, 24], return_df=True)
    print(lb['lb_pvalue'].values)

Rapporterar modellspec, BIC och LB p-värde lag 6,12,24.
Första skattningen av modellerna gav:

Modell 1 med covid-dummy.
(0,0,0)(1,0,1)[12]: BIC:-3371.60
[0.00342473 0.0136766  0.04629349]

Modell 1.2 utan covid-dummy
(0,0,0)x(1,0,1)[12]: BIC:-3339.85
[4.36779750e-06 3.91159857e-05 4.64653835e-04]

modell 2 utan covid-dummy
(0,0,0)(2,0,2)[12]: BIC:-3339.74
[8.16117050e-06 7.17680601e-05 3.32166836e-03]

modell 3 utan covid-dummy
(1,0,0)(1,0,2)[12]: BIC:-3337.06
[1.64650716e-05 1.63061629e-04 4.10982485e-03]

Givet resultaten i modell 1 vs 1.2 så kan vi se att covid-dummyn är motiverad för att förbättra modellen.
Tar bort modell 1.2 och skattar en andra gång med covid-dummy på alla.

Modell 1
(0,0,0)x(1,0,1)[12]: BIC:-3371.60
[0.00342473 0.0136766  0.04629349]

Modell 2
(0,0,0)(2,0,2)[12]: BIC:-3369.16
[0.00345665 0.01194988 0.04887431]

Modell 3
(1,0,0)(1,0,2)[12]: BIC:-3359.59
[0.00427461 0.00882542 0.01261009]

För utvärderingen, som görs med en expanding window (alternativt rolling window) pseudo out-of-sample forecast eval, tar vi med oss modell 1 och 3 från skattning och adderar även en vanlig AR(1) som var den modell som presterade bäst på rullande 12m-serien. 
    - Benchmarks är samma som förut: Naive, Mean.

# Pseudo out-of-sample och prognosutvärdering med modeller.

In [ ]:
# Expanding window pseudo out-of-sample forecast

results_MoM = {}

for h in forecast_horizons:

    actuals = []
    preds_ar1 = []
    preds_model_1 = []
    preds_model_3 = []
    preds_mean = []
    preds_naive = []

    evaluation_length = len(KPIF_MoM_trimmed.loc[train_end:]) - h + 1

    for t in range(evaluation_length):

        train = KPIF_MoM_trimmed.iloc[:len(KPIF_MoM_trimmed.loc[:train_end]) + t]
        cd = covid_dummy[:len(train)]

        raw_future = covid_dummy[len(train):len(train)+h]
        future_exog = np.zeros(h)
        future_exog[:len(raw_future)] = raw_future
        future_exog = future_exog.reshape(-1, 1)

        # AR(1)
        ar1 = ARIMA(train,
                    order=(1,0,0),
                    exog=cd).fit(method_kwargs={'maxiter':200})
        fc_ar1 = ar1.forecast(steps=h, exog=future_exog)

        # Model 1
        model_1 = SARIMAX(train,
                          order=(0,0,0),
                          seasonal_order=(1,0,1,12),
                          exog=cd).fit(disp=False, maxiter=200)
        fc_model_1 = model_1.forecast(steps=h, exog=future_exog)

        # Model 3
        model_3 = SARIMAX(train,
                          order=(1,0,0),
                          seasonal_order=(1,0,2,12),
                          exog=cd,
                          trend='c').fit(disp=False, maxiter=200)
        fc_model_3 = model_3.forecast(steps=h, exog=future_exog)
        
        actual_idx = len(train) + h -1

        if actual_idx < len(KPIF_MoM_trimmed):
            preds_ar1.append(fc_ar1.iloc[-1])
            preds_model_1.append(fc_model_1.iloc[-1])
            preds_model_3.append(fc_model_3.iloc[-1])
            preds_naive.append(train.iloc[-1, 0])
            preds_mean.append(train.iloc[:, 0].mean())
            actuals.append(KPIF_MoM_trimmed.iloc[actual_idx, 0])

    a = np.array(actuals)
    r = np.array(preds_ar1)
    s1 = np.array(preds_model_1)
    s3 = np.array(preds_model_3)

    results_MoM[f'h={h}'] = {
        'AR1_RMSE': np.sqrt(np.mean((a-r)**2)),   'AR1_MAE': np.mean(np.abs(a-r)),
        'MODEL_1_RMSE':  np.sqrt(np.mean((a-s1)**2)),  'M1_MAE':  np.mean(np.abs(a-s1)),
        'MODEL_3_RMSE':  np.sqrt(np.mean((a-s3)**2)),  'M3_MAE':  np.mean(np.abs(a-s3)),
        'NAIVE_RMSE': np.sqrt(np.mean((a-np.array(preds_naive))**2)),
        'NAIVE_MAE':  np.mean(np.abs(a-np.array(preds_naive))),
        'MEAN_RMSE':  np.sqrt(np.mean((a-np.array(preds_mean))**2)),
        'MEAN_MAE':   np.mean(np.abs(a-np.array(preds_mean))),
        'e_ar1':     a - r,
        'e_model_1': a - s1,
        'e_model_3': a - s3
    }

print(pd.DataFrame(results_MoM).T)

In [ ]:
# DM-test
print("DM-test: AR(1) vs Model 1 och Model 3 (MoM-serien)")
print("-" * 55)

for h in forecast_horizons:
    e_r  = results_MoM[f'h={h}']['e_ar1']
    e_s1 = results_MoM[f'h={h}']['e_model_1']
    e_s3 = results_MoM[f'h={h}']['e_model_3']
    
    for label, e_s in [('Model 1', e_s1), ('Model 3', e_s3)]:
        d = e_s**2 - e_r**2  # positivt = AR(1) bättre
        T = len(d)
        
        dm_stat = d.mean() / (np.std(d, ddof=1) / np.sqrt(T))
        p_dm = 2 * (1 - stats.norm.cdf(np.abs(dm_stat)))
        
        hln_stat = dm_stat * np.sqrt((T + 1 - 2*h + h*(h-1)/T) / T)
        p_hln = 2 * (1 - stats.t.cdf(np.abs(hln_stat), df=T-1))
        
        stars_dm  = "***" if p_dm  < 0.01 else "**" if p_dm  < 0.05 else "*" if p_dm  < 0.10 else ""
        stars_hln = "***" if p_hln < 0.01 else "**" if p_hln < 0.05 else "*" if p_hln < 0.10 else ""
        
        print(f"h={h:2d} AR1 vs {label} | DM={dm_stat:+.3f} p={p_dm:.3f}{stars_dm:3s} | HLN={hln_stat:+.3f} p={p_hln:.3f}{stars_hln:3s}")

Resultat från första utvärderingen:

      AR1_RMSE   AR1_MAE  MODEL_1_RMSE    M1_MAE  MODEL_3_RMSE    M3_MAE  \
h=1   0.003252  0.002485      0.004301  0.003515      0.004098  0.003395   
h=3   0.003278  0.002533      0.004251  0.003418      0.004049  0.003365   
h=6   0.003469  0.002773      0.004356  0.003590      0.004498  0.003741   
h=12  0.003490  0.002790      0.004527  0.003633      0.004412  0.003585   

      NAIVE_RMSE  NAIVE_MAE  MEAN_RMSE  MEAN_MAE  
h=1     0.005162   0.004086   0.003281  0.002492  
h=3     0.004826   0.003642   0.003258  0.002479  
h=6     0.005837   0.004742   0.003449  0.002722  
h=12    0.004145   0.003338   0.003461  0.002732  

Eftersom vi i loopen endast sparar sista värdet i prognosbanan för h=1,3,6,12 behöver vi justera koden för att kunna transformera till 12m rullande inflation och jämföra med första analysen.
AR(1), bortsett från benchmark "Mean", presterade bäst bland modellerna och är den vi kommer att använda och jämföra med tidigare analys när vi rekonstruerat 12m rullande inflation.
Ett återkommande fenomen tycks vara att den modell som enligt auto_arima, eller snarare enligt statistika, ska vara den bästa, inte lyckas leva upp till förväntningarna. För inflation tycks AR(1) vara den dynamiska modell som fångar dynamiken bäst bland alla de modeller vi testat.
      - Möjligt att den kan förbättras mer med fler exogena variabler, men det får testas vid ett annat tillfälle.

Eftersom AR(1) presterade bäst i båda analyserna ska det bli väldigt intressant att jämföra resultaten efter att MoM-datan transformerats till rullande 12m i en pseudo out-of-forecast evaluation loop. Men först behöver vi ytterligare data för att jämförbar första månad ska kunna beräknas på MoM.

In [ ]:
# Skapar MoM på hela serien för att kunna rekonstruera första obs av 12m rullande på MoM
KPIF_MoM_full = inf_data.set_index('period')[['KPIF-MoM']].dropna()
KPIF_MoM_full.index = pd.DatetimeIndex(KPIF_MoM_full.index, freq='MS')
KPIF_MoM_full = KPIF_MoM_full.loc[:'2026-05-01'] 

# Anpassar dummy till hela serien
covid_dummy_full = ((KPIF_MoM_full.index >= '2021-07-01') & 
                    (KPIF_MoM_full.index <= '2023-12-01')).astype(int)

# Rekonstruerar 12m rullande från MoM
# Jämför med analys 1

In [ ]:
# Pseudo out-of-sample forecast evaluation
# AR(1) skattas på MoM-serien.
# h månaders prognoser rekonstrueras sedan till 12-månadersinflation.
# h = 1, dopad MoM transformering pga 11 kända obs. h-steg därefter inkluderar fler prognosticerade obs i transformering. (samma för rullande 12m)
# h = 1 innehåller alltid 11/12 känd information per definition. 

forecast_horizons = [1, 3, 6, 12]
train_end = '2023-12-01'


for train_end in ['2023-01-01', '2023-06-01', '2023-12-01']: # upd (additional train periods) for comparison with first analysis.

    results_reconstructed = {}

    for h in forecast_horizons:

        actuals = []
        preds_reconstructed = []

        failed_ar1 = 0

        evaluation_length = len(KPIF_trimmed.loc[train_end:]) - h + 1

        for t in range(evaluation_length):

            # Expanding window på MoM-serien
            train = KPIF_MoM_full.iloc[:len(KPIF_MoM_full.loc[:train_end]) + t]
            cd = covid_dummy_full[:len(train)]

            # Framtida covid-dummy
            raw_future = covid_dummy_full[len(train):len(train)+h]

            future_exog = np.zeros(h)
            future_exog[:len(raw_future)] = raw_future
            future_exog = future_exog.reshape(-1,1)

            # Skatta AR(1)
            ar1 = ARIMA(
                train,
                order=(1,0,0),
                exog=cd
            ).fit(method_kwargs={'maxiter':200})

            if not ar1.mle_retvals['converged']:
                failed_ar1 += 1

            # Prognosticera h månader framåt (MoM)
            fc = ar1.forecast(
                steps=h,
                exog=future_exog
            )


            if h < 12:
                last_known = train.iloc[-(12-h):, 0].values
            else:
                last_known = np.array([])

            # Kombinera observerade + prognosticerade månader
            all_12 = np.concatenate([
                last_known,
                fc.values
            ])

            # Rekonstruera 12-månadersinflation
            recon_12 = np.prod(1 + all_12) - 1

            # Samma actual-index som tidigare analys
            actual_idx = len(KPIF_trimmed.loc[:train_end]) + t + h - 1

            if actual_idx < len(KPIF_trimmed):

                preds_reconstructed.append(recon_12)
                actuals.append(
                    KPIF_trimmed.iloc[actual_idx,0]
                )

        a = np.array(actuals)
        r = np.array(preds_reconstructed)

        results_reconstructed[f'h={h}'] = {
            'RMSE' : np.sqrt(np.mean((a-r)**2)),
            'MAE'  : np.mean(np.abs(a-r)),
            'errors' : a-r
        }

    print(
        pd.DataFrame(results_reconstructed).T.drop(columns=['errors'])
    )

In [ ]:
# Jämför AR(1), utfall 12m rekonstruerad från MoM vs 12m rullande
print(f"{'':8} {'MoM_RMSE':>12} {'MoM_MAE':>10} | {'Direct_AR1_RMSE':>16} {'Direct_AR1_MAE':>14}")
for h in forecast_horizons:
    mom_rmse = results_reconstructed[f'h={h}']['RMSE']
    mom_mae  = results_reconstructed[f'h={h}']['MAE']
    dir_rmse = results[f'h={h}']['AR1_RMSE']
    dir_mae  = results[f'h={h}']['AR1_MAE']
    print(f"h={h:2d}   {mom_rmse:>12.6f} {mom_mae:>10.6f} | {dir_rmse:>16.6f} {dir_mae:>14.6f}")

    Robust och Känslighetsanalys

För att undersöka hur valet av dataserie påverkar prognoskvaliteten genomförs en robusthet och känslighetsanalys. Två alternativa längder på träningsdatan adderas till testen där vi kör modellerna på 12m-data och MoM-data. De alternativa avskärningarna vi gör, mitt i eller nära slutet av covid-chocken, är nödvändigtvis inte de bästa perioderna, men eftersom jämförelsen är konsekvent över modellerna så är resultaten jämförbara.

    De tre avskärningspunkterna:
train_end = 2023-01
train_end = 2023-06
train_end = 2023-12 

När vi kör de tre iterationerna med 12m-datan kan vi se att de olika modellerna klara av att anpassa sig i varierande grad till rådande regim. h=1 till 3 presterar fortfarande hyfsat bra i kontrast till Naive och Mean som genererar stora avvikelser under covid-perioden. Men prestationen är troligtvis en fråga om persistens, precis som tidigare noterat, där båda modeller, och benchmarks kan tillskriva en stor del av prestationen till att, bortsett från chocker, så rör sig serien relativt lite. När modellerna börjar prognosticera i normaliseringen efter inflationschocken ~2023-01 så lyckas de inte följa med när serien återgår mot sitt medelvärde, vilket syns i att SARIMAX och AR(1) försämras kraftigt på längre horisonter medan mean presterar mycket bättre.

Mellan SARIMAX och AR(1) tycks resultaten relativt sett vara väldigt konsekventa. En viss variation för resultaten i train_end = 2023-01 där SARIMAX presterar bättre på h=1 och h=3, men sämre (som i de andra iterationerna) vid h=6 och 12.

Eftersom AR(1) är den modell som överlag presterat bäst är det även den som används vid jämförelse av prognoser på 12m-KPIF och MoM-rekonstruerad 12m. Får vi någon skillnad i prognoskvalitet om vi prognosticerar på MoM-data och sedan rekonstruerar rullande 12m-inflationsdata? Är prognoser på MoM-data bättre/sämre än de på 12m-data?

När vi jämför resultaten på AR(1) från rullande 12m KPIF och MoM 12m rekonstruerad så kan vi se att det finns vissa skillnader i prognosprestation vid olika h. AR(1) på MoM-datan presterar mycket bättre vid lägre h och i vissa fall även vid h=6, men inte lika bra på längre horistoner h=12. I train_end = 2023-01 presterade däremot AR(1) på MoM-datan bättre på alla h-steg. I train_end = 2023-06 och 2023-12 kan vi dock se att AR(1) på 12m-datan tycks börja prestera likvärdigt och även bättre på h=6 och h=12.

    - Rekonstruerad MoM var konsekvent bättre än direkt AR(1) på rullande 12m för train_end = 2023-01 (alla punkter). En möjlig förklaring är att MoM tycks bättre fånga den rörelsen i de individuella prognoerna som sedan sammansätts. AR(1)-modellen på 12m-datan är troligtvis för trögrörlig för att kunna anpassa sig till denna period där inflationen är mer volatil. Detta är troligtvis också anledning till att AR(1) på 12m-data presterar bättre i train_end = 2023-12 jämfört med de tidigare iterationerna. Vi vet att AR-koefficienten hade ett värde nära 1, vilket kan vara en styrka för modellen om det inte händer så mycket (som när serien återvänt nära mean efter chocken), men inte när serien rör sig snabbt i en riktning.

    - MoM-data, "as is" eller rekonstruerad tycks överlag vara bättre för prognoser på kortare tidshorisont, medan AR(1) på 12m rullande-inflationsdata ser, enligt denna utvärdering, ut att ge bättre prognoser för tidshorisonter längre ut (h=12). Detta är även i linje med vad Marcellino et al. (2006) beskriver som bias-variance tadeoff mellan itererade och direkta prognoser. MoM som är itererad kan hantera och anpassa sig till snabba rörelser bättre, men ackumulerar samtidigt fel på lång sikt i större grad. En AR(1) direkt på 12m är beviserligen mer stabil under normala/lugna perioder.
    

* Rullande 12m KPIF
train_end = 2023-01
     SARIMAX_RMSE SARIMAX_MAE <AR1_RMSE  <AR1_MAE NAIVE_RMSE NAIVE_MAE  MEAN_RMSE  MEAN_MAE
h=1      0.004611    0.003506  0.005835  0.004436   0.006068  0.004617   0.023682  0.014453  
h=3      0.008788    0.006934  0.010126  0.007967   0.011173  0.008946   0.018497  0.011661 
h=6      0.013945    0.012084  0.016483  0.012797   0.019451  0.015372   0.012414  0.008427  
h=12     0.025823    0.024223  0.023783  0.016436    0.03012  0.022017   0.006665  0.005467

train_end = 2023-06
     SARIMAX_RMSE SARIMAX_MAE <AR1_RMSE  <AR1_MAE NAIVE_RMSE NAIVE_MAE  MEAN_RMSE  MEAN_MAE  
h=1      0.004495    0.003348  0.005699  0.004336   0.005778  0.004381   0.012177  0.008278   
h=3      0.008877    0.007048  0.008581  0.006892   0.009349  0.007474   0.008595  0.006616  
h=6      0.014088     0.01207  0.012141  0.009898   0.014483  0.011812   0.006512  0.005351 
h=12     0.025349    0.023571  0.015456  0.011084   0.020104   0.01472   0.006399  0.005238  

train_end = 2023-12 
     SARIMAX_RMSE SARIMAX_MAE <AR1_RMSE  <AR1_MAE NAIVE_RMSE NAIVE_MAE  MEAN_RMSE  MEAN_MAE  
h=1      0.004837    0.003678  0.004765  0.003748   0.004696  0.003749   0.006527  0.005345  
h=3      0.008938    0.007138  0.006491  0.005488    0.00671  0.005675   0.006167  0.005046 
h=6      0.014249    0.012116  0.009096  0.008166   0.009958  0.008833   0.00646  0.005308
h=12     0.026506    0.025238  0.008889  0.006807   0.010385  0.008283   0.00671  0.005391


* MoM 12m reconstructed KPIF

train_end = 2023-01
          RMSE       MAE 
h=1   0.004013  0.003119
h=3   0.007493  0.006352
h=6   0.013652  0.011359
h=12   0.02333  0.019185

train_end = 2023-06
          RMSE       MAE
h=1   0.003894  0.002912
h=3   0.007228  0.006086
h=6   0.011644  0.009716
h=12  0.018189  0.015012

train_end = 2023-12 
          RMSE       MAE
h=1   0.003649  0.002659
h=3   0.006063  0.005163
h=6   0.009243   0.00786
h=12  0.011693   0.01025

# Prognos MoM - start 2026-05-01

In [ ]:
# Prognosticera inflation med AR(1) från seriens slut 2026-05-01
# Separat skattning från AR(1)-modellen som utvärderades ovan.

ar1fc_mom_model = ARIMA(
    KPIF_MoM_trimmed,
    order=(1,0,0),
    exog=covid_dummy 
).fit()


future_exog = np.zeros((12,1))  


fc_mom = ar1fc_mom_model.get_forecast(
    steps=12,
    exog=future_exog 
)

forecast_mom = fc_mom.predicted_mean   
ci_mom = fc_mom.conf_int()              


print("1 månad:", forecast_mom.iloc[0])     
print("3 månader:", forecast_mom.iloc[2])   
print("6 månader:", forecast_mom.iloc[5])   
print("12 månader:", forecast_mom.iloc[-1])

print("\nHela prognosbanan:")
print(forecast_mom)

print("\n95%-konfidensintervall:")
print(ci_mom)

In [ ]:
# Jämförbara prognosvärden på 12m-inflation
# Hämtar senaste 11 kända MoM-värden
last_11 = KPIF_MoM_full.iloc[-11:, 0].values

# Rekonstruera 12m för varje horisont
for h in [1, 3, 6, 12]:
    if h < 12:
        known = KPIF_MoM_full.iloc[-(12-h):, 0].values
    else:
        known = np.array([])
    
    all_12 = np.concatenate([known, forecast_mom.values[:h]])
    recon = np.prod(1 + all_12) - 1

    print(f"h={h:2d}: rekonstruerad 12m-inflation = {recon:.4f} ({recon*100:.2f}%)")

För att säkerställa jämförbarhet mellan de två ansatserna används AR(1) som prognosmodell i båda fallen. Skillnader i prognosutfall tillskrivs därav datastruktur snarare än modellval.

Jämfört med den rullande 12m-inflationen kan vi se att prognosen på MoM-serien genererar en prognosbana något lägre runt 1,55%-1,75% i årstakt. Bortsett från den initiala prognosen 2026-06-01 så tycks modellens mean-prognos konvergera till 0,1326% i MoM inflation på grund av modellstrukturen och dess ovillkorliga medelvärde.

Skillnaderna i utfall mellan de två metoderna härleds främst till datastrukturen som respektive prognos är baserad på. Modellen som är skattad på den rullande 12m-inflationen extrapolerar en redan utjämnad och mycket persistent serie, vilket leder till en prognos som landar på 1,8%. Att det är nära riksbankens inflationsmål på 2% bör inte tolkas som att prognosförmågan nödvändigtvis är högre, utan att seriens struktur och träningsdatans horisont leder till och premierar hög persistens givet att observationer är mean reverting. 

Båda prognoserna är rimliga sett till det ekonomiska läget och att KPIF tenderat understiga 2%-målet under längre perioder. Riksbankens agerande signalerar även att man förväntar sig en återgång till tidigare regim (från närmaste årens chock) med milt fallande räntor, vilket självklart påverkar prognosbanor.  

# Övrigt
Hade jag kört på den otrimmade serien med start 1980 hade ADF troligtvis inte kunnat förkasta H0. Den otrimmade serien har en tydlig nedåtgående trend från ~10% på 80-talet till ~2% efter 1995 och hade varit svår att modellera. Det hade troligtivs behövts diffa serien en gång till I(1) likt Stock & Watson i "WHY HAS U.S. INFLATION BECOME HARDER TO FORECAST". Detta hade troligtvis lett till ett helt annat modellval.
    - Valet av att trimma till 1994 utnyttjar det faktum att Riksbanken introducerade inflationsmålet 1995 och att serien därefter beter sig stationärt kring ~2%.
    - Unobserved Components Stochastic Volatility (UCSV) model hade varit intressant att testa givet diagnostikproblemen: heteroskedasticitet, JB förkastar och LB förkastar. UC-SV hanterar heteroskedasticiteten direkt genom tidvarierande varians, vilket är vad covid-chocken kanske kräver.

"A comparison of direct and iterated multistep AR methods for forecasting macroeconomic time series - (Massimiliano Marcellino, James H. Stock, Mark W. Watson)
    . in theory, iterated forecasts are more efficient if the one-period ahead model is correctly specified, but direct forecasts are more robust to model misspecification.
    . The iterated forecasts typically outperform the direct forecasts, particularly, if the models can select long-lag specifications. The relative performance of the iterated forecasts improves with the forecast horizon."



# Om tid finns, eller till nästa prognosarbete
* Cross-validation genom att dela datan i 4 delar. Träna på 3 av dessa och prognosticera den 4e. Ger 4st utvärderingar på samma mängd data och insikt i hur modellen presterar på hela datasettet.